# Simulación del juego Nim

## Introducción

En este notebook exploraremos diferentes estrategias para simular la toma de decisiones en el juego Nim. Nim es un juego estratégico en el que dos jugadores se turnan para retirar palos de un montón.

Reglas:
 - Se comienza con un número determinado de palos.
 - En cada turno, el jugador puede retirar entre 1 y un máximo de palos permitido.
 - Los jugadores alternan turnos hasta que no queden palos.
 - Pierde quien tome el último palo.

In [4]:
import sys
import os
import pandas as pd
import random
import numpy as np

# Fijar la semilla
SEED = 123
random.seed(SEED)
np.random.seed(SEED)

# Agregar los directorios al path para poder importar los módulos
sys.path.append(os.path.abspath("../games"))
sys.path.append(os.path.abspath("../methods"))
sys.path.append(os.path.abspath("../selection"))
sys.path.append(os.path.abspath("../graphviz"))

# Importar la clase NimGame y las funciones de los métodos
from nim import NimGame
from monte_carlo import MCPlay, MCAgent
from monte_carlo_tree_search import MCTS, MCTSPlay, Node, MCTSAgent
from epsilon_greedy import EpsilonGreedy
from softmax import Softmax
from adaptive_softmax import AdaptiveSoftmax
from ucb1 import UCB1
from ucb2 import UCB2
from gradiente_preferencias import GradienteDePreferencias
from MCTS_graphviz import generate_tree_MCTS

## Implementación de la clase `NimGame`

En esta sección, se explica cómo ha sido implementada la clase `NimGame` que modela el juego. Para ello, a continuación se presenta una breve descripción de cada uno de los métodos que la componen:

 - `__init__(self, initial_sticks=21, max_sticks_per_turn=4, starting_player=1)`: Se utiliza para inicializar el juego. Sus parámetros incluyen:
   - `initial_sticks`: número de palos iniciales.
   - `max_sticks_per_turn`: número máximo de palos que un jugador puede retirar por turno.
   - `starting_player`: el jugador que empieza el juego (1 o -1).

 - `valid_actions(self)`: Este método devuelve las acciones válidas que un jugador puede realizar, es decir, el número de palos que puede retirar.

 - `action(self, amount)`: Este método simula un turno del juego, restando el número especificado de palos a los palos restantes y pasándole el turno al jugador siguiente, y devuelve el nuevo estado del juego. Además, realiza una comprobación para asegurarse de que la acción solicitada sea válida.

 - `terminal(self)`: Verifica si el juego ha terminado, esto es, si no quedan palos.

 - `winner(self)`: Devuelve quién es el ganador. Si el juego ha terminado, devuelve el número del jugador ganador (1 o -1), pero si no ha terminado devuelve 0.

 - `draw(self)`: Este método dibuja el estado actual del juego, imprimiendo cuántos palos quedan.

## Simulaciones

En esta sección, se realizan simulaciones con distintas estrategias para la toma de decisiones en el juego. Comenzamos creando una instancia del juego `NimGame`, y visualizamos su estado inicial. Por defecto, el juego comenzará con 21 palos, y en cada turno, el jugador correspondiente podrá retirar como máximo 4 palos. Sin embargo, estos parámetros se pueden modificar con `initial_sticks` y `max_sticks_per_turn`.

In [2]:
# Crear una instancia del juego Nim con los valores predeterminados
game = NimGame()

# Mostrar el estado inicial del juego
game.draw()

Quedan 21 palos.


A continuación, se presentan los distintos métodos utilizados para realizar las simulaciones.

### Monte-Carlo

En primer lugar, se implementa el método de Monte-Carlo para estimar las mejores jugadas. 

In [2]:
# Crear una instancia del juego Nim
game = NimGame(initial_sticks=21, max_sticks_per_turn=4, starting_player=-1)

# Mostrar el estado inicial del juego
game.draw()

# Definir el número de simulaciones por movimiento
num_simulations = 1000

# Bucle del juego hasta que termine
while not game.terminal():
    if game.turn == 1:  # Turno del jugador humano
        print("\nTu turno...")
        jugada = int(input("¿Cuántos palos quieres retirar? "))

        if jugada in game.valid_actions():
            game = game.action(jugada)
        else:
            print("Jugada inválida. Inténtalo de nuevo.")
            continue  # Volver a pedir jugada

    else:  # Turno de la IA (Monte-Carlo)
        print("\nTurno de la computadora...")
        game = MCPlay(game, num_simulations)

    game.draw()  # Mostrar el estado después del turno

# Anunciar el ganador
if game.winner() == 1:
    print("\n¡Ganas la partida! 🎉")
else:
    print("\nLa computadora gana. ¡Inténtalo de nuevo! 🤖")

Quedan 21 palos.

Turno de la computadora...
Quedan 17 palos.

Tu turno...
Quedan 16 palos.

Turno de la computadora...
Quedan 13 palos.

Tu turno...
Quedan 11 palos.

Turno de la computadora...
Quedan 10 palos.

Tu turno...
Quedan 8 palos.

Turno de la computadora...
Quedan 5 palos.

Tu turno...
Quedan 2 palos.

Turno de la computadora...
Quedan 0 palos.

La computadora gana. ¡Inténtalo de nuevo! 🤖


### Monte-Carlo Tree Search

En esta sección, se utiliza el método de Monte-Carlo Tree Search para estimar las mejores jugadas en el juego Nim. Este algoritmo está compuesto por distintas fases, que son:

 1. **Selección:** En esta fase, se parte del nodo raíz y se desciende por el árbol seleccionando nodos hijos sucesivamente según una estrategia. Esto continúa hasta llegar a un nodo hoja, es decir, un nodo que tiene al menos un hijo potencial al que no se le ha aplicado ninguna simulación todavía. Entre las estrategias de selección encontramos: $\epsilon$-greedy, Softmax, Adaptive Softmax, UCB1, UCB2 y Gradiente de Preferencias.

 2. **Expansión:** A partir del nodo hoja seleccionado, se genera un nuevo nodo hijo, es decir, se aplica un movimiento válido que aún no ha sido explorado desde el nodo hoja.

 3. **Simulación:** Desde el nodo hijo recién creado, se completa una jugada aleatoria hasta alcanzar un estado terminal (por ejemplo, una victoria, derrota o empate). Mediante esta simulación, se puede estimar el resultado potencial de seguir esa línea de decisión.

 4. **Retropropagación:** Los resultados de la simulación se propagan hacia atrás, usándose para actualizar las estadísticas de los nodos, que son recorridos desde el nodo expandido hasta el nodo raíz. Esto permite que en futuras decisiones se refuercen las rutas más prometedoras y se descarten las menos efectivas.

In [ ]:
# Crear una instancia del juego Nim
game = NimGame(initial_sticks=21, starting_player=1)

# Mostrar el estado inicial del juego
game.draw()

# Definir el número de simulaciones por movimiento
num_simulations = 1000

# Escoger el método de selección
selection_algorithm_class = EpsilonGreedy

# Bucle del juego hasta que termine
while not game.terminal():
    if game.turn == 1:  # Turno del jugador humano
        print("\nTu turno...")
        jugada = int(input("¿Cuántos palos quieres retirar? "))

        if jugada in game.valid_actions():
            game = game.action(jugada)
        else:
            print("Jugada inválida. Inténtalo de nuevo.")
            continue

    else:  # Turno de la IA con MCTS
        print("\nTurno de la computadora...")
        game = MCTSPlay(game, num_simulations, selection_algorithm_class, epsilon = 0.8)

    game.draw()

# Resultado
if game.winner() == 1:
    print("\n¡Ganas la partida! 🎉")
else:
    print("\nLa computadora gana. ¡Inténtalo de nuevo! 🤖")

Quedan 21 palos.

Turno de MC...
Quedan 18 palos.

Turno de la computadora...
Quedan 16 palos.

Turno de MC...
Quedan 15 palos.

Turno de la computadora...
Quedan 12 palos.

Turno de MC...
Quedan 10 palos.

Turno de la computadora...
Quedan 6 palos.

Turno de MC...
Quedan 5 palos.

Turno de la computadora...
Quedan 3 palos.

Turno de MC...
Quedan 0 palos.

¡Ganas la partida! 🎉


A continuación, se utiliza `graphviz` como herramienta de depuración, generando un diagrama con el árbol de posiciones siguientes a partir de una determinada posición inicial, dada por un nodo raíz. Cada nodo del árbol representa un estado del juego, el cual se alcanza mediante una secuencia de movimientos, y contiene información como el turno del jugador, el número de visitas y la recompensa de cada jugador. Las aristas indican las acciones tomadas para pasar de un estado al siguiente. Se puede especificar la profundidad máxima del árbol generado.

In [ ]:
# Crear estado inicial del juego
initial_state = NimGame(initial_sticks=5, max_sticks_per_turn=3, starting_player=1)

# Visualizar el árbol de MCTS
root_node = Node(initial_state, None)
mcts = MCTS(root_node, Softmax, simulations=1000)
mcts.run()
id_to_node = generate_tree_MCTS(root_node, max_depth=2, filename="depuration_trees/arbol_nim_2", format="jpg")

Los nodos en el árbol aparecen identificados mediante un ID. Para poder visualizar cuál es el estado de cada nodo, la función `generate_tree_MCTS` devuelve un diccionario que asocia cada ID al nodo correspondiente.

In [17]:
node = id_to_node['nodo3']

node.state.draw()

Quedan 4 palos.


## Simulaciones comparativas

In [5]:
def play_comparative_game(agent1, agent2, initial_sticks=21, max_sticks_per_turn=4, starting_player=1):
    game = NimGame(initial_sticks=initial_sticks, max_sticks_per_turn=max_sticks_per_turn, starting_player=starting_player)
    agents = {1: agent1, -1: agent2}

    while not game.terminal():
        game = agents[game.turn].move(game)

    return game.winner()

In [ ]:
def run_simulations(agent_1, agent_2, initial_sticks=21, max_sticks_per_turn=4, n_games=100):
    results = []

    for i in range(n_games):
        # Alternar jugador inicial
        starting_player = 1 if i % 2 == 0 else -1

        # Asignar agentes según quién empieza
        if starting_player == 1:
            winner = play_comparative_game(agent_1, agent_2, initial_sticks, max_sticks_per_turn, starting_player)
        else:
            winner = play_comparative_game(agent_2, agent_1, initial_sticks, max_sticks_per_turn, starting_player)
            # Invertir perspectiva
            winner *= -1

        results.append(winner)

    df = pd.DataFrame(results, columns=["winner"])
    win_rate = (df["winner"] == 1).mean()
    print(f"% de partidas ganadas por el agente 1: {win_rate:.2%} ({df['winner'].value_counts().to_dict()})")
    return df

In [23]:
agent_mc = MCAgent(1000)
agent_mcts_eps = MCTSAgent(1000, EpsilonGreedy, epsilon = 0.2)
agent_mcts_UCB1 = MCTSAgent(1000, UCB1, c=1)
agent_mcts_UCB2 = MCTSAgent(1000, UCB2, alpha=0.5)
agent_mcts_soft = MCTSAgent(1000, Softmax, tau = 1)
agent_mcts_adapsoft = MCTSAgent(1000, AdaptiveSoftmax, tau_0 = 1, alpha = 0.5)
agent_mcts_grad = MCTSAgent(1000, GradienteDePreferencias, alpha = 0.2)

### Monte Carlo vs MCTS con $\epsilon$-greedy

In [7]:
df_mc_mcts_eps = run_simulations(agent_mc, agent_mcts_eps, initial_sticks=21, max_sticks_per_turn=4, n_games=100)

% de partidas ganadas por el agente 1: 44.00% ({-1: 56, 1: 44})


### Monte Carlo vs MCTS con UCB1

In [ ]:
df_mc_mcts_UCB1 = run_simulations(agent_mc, agent_mcts_UCB1, initial_sticks=21, max_sticks_per_turn=4, n_games=100)

% de partidas ganadas por el agente 1: 32.00% ({-1: 68, 1: 32})


### Monte Carlo vs MCTS con UCB2

In [15]:
df_mc_mcts_UCB2 = run_simulations(agent_mc, agent_mcts_UCB2, initial_sticks=21, max_sticks_per_turn=4, n_games=100)

% de partidas ganadas por el agente 1: 84.00% ({1: 84, -1: 16})


### Monte Carlo vs MCTS con Softmax

In [18]:
df_mc_mcts_soft = run_simulations(agent_mc, agent_mcts_soft, initial_sticks=21, max_sticks_per_turn=4, n_games=100)

% de partidas ganadas por el agente 1: 71.00% ({1: 71, -1: 29})


### Monte Carlo vs MCTS con Softmax Adaptativo

In [19]:
df_mc_mcts_adapsoft = run_simulations(agent_mc, agent_mcts_adapsoft, initial_sticks=21, max_sticks_per_turn=4, n_games=100)

% de partidas ganadas por el agente 1: 66.00% ({1: 66, -1: 34})


### Monte Carlo vs MCTS con Gradiente de Preferencias

In [24]:
df_mc_mcts_grad = run_simulations(agent_mc, agent_mcts_grad, initial_sticks=21, max_sticks_per_turn=4, n_games=100)

% de partidas ganadas por el agente 1: 95.00% ({1: 95, -1: 5})


### MCTS con $\epsilon$-greedy vs MCTS con UCB1

In [ ]:
df_mcts_eps_UCB1 = run_simulations(agent_mcts_eps, agent_mcts_UCB1, initial_sticks=21, max_sticks_per_turn=4, n_games=100)

% de partidas ganadas por el agente 1: 27.00% ({-1: 73, 1: 27})


### MCTS con $\epsilon$-greedy vs MCTS con UCB2

In [ ]:
df_mcts_eps_UCB2 = run_simulations(agent_mcts_eps, agent_mcts_UCB2, initial_sticks=21, max_sticks_per_turn=4, n_games=100)

% de partidas ganadas por el agente 1: 89.00% ({1: 89, -1: 11})


### MCTS con $\epsilon$-greedy vs MCTS con Softmax

In [ ]:
df_mcts_eps_soft = run_simulations(agent_mcts_eps, agent_mcts_soft, initial_sticks=21, max_sticks_per_turn=4, n_games=100)

% de partidas ganadas por el agente 1: 65.00% ({1: 65, -1: 35})


### MCTS con $\epsilon$-greedy vs MCTS con Softmax Adaptativo

In [ ]:
df_mcts_eps_adapsoft = run_simulations(agent_mcts_eps, agent_mcts_adapsoft, initial_sticks=21, max_sticks_per_turn=4, n_games=100)

% de partidas ganadas por el agente 1: 66.00% ({1: 66, -1: 34})


### MCTS con $\epsilon$-greedy vs MCTS con Gradiente de Preferencias

In [ ]:
df_mcts_eps_grad = run_simulations(agent_mcts_eps, agent_mcts_grad, initial_sticks=21, max_sticks_per_turn=4, n_games=100)

% de partidas ganadas por el agente 1: 95.00% ({1: 95, -1: 5})


### MCTS con UCB1 vs MCTS con UCB2

In [ ]:
df_mcts_UCB1_UCB2 = run_simulations(agent_mcts_UCB1, agent_mcts_UCB2, initial_sticks=21, max_sticks_per_turn=4, n_games=100)

% de partidas ganadas por el agente 1: 89.00% ({1: 89, -1: 11})


### MCTS con UCB1 vs MCTS con Softmax

In [ ]:
df_mcts_UCB1_soft = run_simulations(agent_mcts_UCB1, agent_mcts_soft, initial_sticks=21, max_sticks_per_turn=4, n_games=100)

% de partidas ganadas por el agente 1: 83.00% ({1: 83, -1: 17})


### MCTS con UCB1 vs MCTS con Softmax Adaptativo

In [ ]:
df_mcts_UCB1_adapsoft = run_simulations(agent_mcts_UCB1, agent_mcts_adapsoft, initial_sticks=21, max_sticks_per_turn=4, n_games=100)

% de partidas ganadas por el agente 1: 87.00% ({1: 87, -1: 13})


### MCTS con UCB1 vs MCTS con Gradiente de Preferencias

In [32]:
df_mcts_UCB1_grad = run_simulations(agent_mcts_UCB1, agent_mcts_grad, initial_sticks=21, max_sticks_per_turn=4, n_games=100)

% de partidas ganadas por el agente 1: 98.00% ({1: 98, -1: 2})


### MCTS con UCB2 vs MCTS con Softmax

In [33]:
df_mcts_UCB2_soft = run_simulations(agent_mcts_UCB2, agent_mcts_soft, initial_sticks=21, max_sticks_per_turn=4, n_games=100)

% de partidas ganadas por el agente 1: 30.00% ({-1: 70, 1: 30})


### MCTS con UCB2 vs MCTS con Softmax Adaptativo

In [34]:
df_mcts_UCB2_adapsoft = run_simulations(agent_mcts_UCB2, agent_mcts_adapsoft, initial_sticks=21, max_sticks_per_turn=4, n_games=100)

% de partidas ganadas por el agente 1: 17.00% ({-1: 83, 1: 17})


### MCTS con UCB2 vs MCTS con Gradiente de Preferencias

In [35]:
df_mcts_UCB2_grad = run_simulations(agent_mcts_UCB2, agent_mcts_grad, initial_sticks=21, max_sticks_per_turn=4, n_games=100)

% de partidas ganadas por el agente 1: 79.00% ({1: 79, -1: 21})


### MCTS con Softmax vs MCTS con Softmax Adaptativo

In [36]:
df_mcts_soft_adapsoft = run_simulations(agent_mcts_soft, agent_mcts_adapsoft, initial_sticks=21, max_sticks_per_turn=4, n_games=100)

% de partidas ganadas por el agente 1: 50.00% ({-1: 50, 1: 50})


### MCTS con Softmax vs MCTS con Gradiente de Preferencias

In [37]:
df_mcts_UCB2_grad = run_simulations(agent_mcts_soft, agent_mcts_grad, initial_sticks=21, max_sticks_per_turn=4, n_games=100)

% de partidas ganadas por el agente 1: 94.00% ({1: 94, -1: 6})
